# Week 2 Lab — Data Validation & Training-Serving Skew

**Objective:** Build data quality checks the way you build health checks for a service.
Then break things on purpose to see why this matters.

What we will do:
1. Load a clean dataset and train a baseline model
2. Write data validation rules (schema checks, value ranges, null checks)
3. Introduce data quality issues and watch model predictions degrade
4. Demonstrate training-serving skew — the silent killer
5. Map each validation to a future production alert

**Operator mindset throughout:** Every check here becomes an alert rule in production.

---
## Part 1 — Baseline: Clean Data, Working Model

First, let's establish what "healthy" looks like. This is your golden signal baseline.
Same idea as knowing your normal CPU/memory before you can detect anomalies.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the Wine dataset — slightly more complex than Iris,
# with features that map well to real-world tabular data
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

print(f'Dataset: {df.shape[0]} samples, {df.shape[1] - 1} features')
print(f'Target classes: {list(wine.target_names)}')
print(f'\n--- Feature summary (this is your healthy data baseline) ---')
df.describe().round(2)

In [ ]:
# Record the "contract" — what healthy data looks like
# This is your schema definition, equivalent to an OpenAPI spec for your data

EXPECTED_SCHEMA = {
    'columns': list(df.columns),
    'dtypes': {col: str(dtype) for col, dtype in df.dtypes.items()},
    'row_count_min': 100,  # minimum expected rows in a batch
    'feature_ranges': {},
}

# Record valid ranges for each feature (like setting thresholds for alerts)
for col in wine.feature_names:
    EXPECTED_SCHEMA['feature_ranges'][col] = {
        'min': float(df[col].min()),
        'max': float(df[col].max()),
        'mean': float(df[col].mean()),
        'std': float(df[col].std()),
    }

print('=== Data Contract (Schema) ===')
print(f'Expected columns: {len(EXPECTED_SCHEMA["columns"])}')
print(f'Expected dtypes: all float64 features + int64 target')
print(f'\nFeature ranges (first 3):')
for col in list(EXPECTED_SCHEMA['feature_ranges'].keys())[:3]:
    r = EXPECTED_SCHEMA['feature_ranges'][col]
    print(f'  {col}: [{r["min"]:.2f}, {r["max"]:.2f}] (mean={r["mean"]:.2f})')

print('\n--- Operator insight ---')
print('This schema IS your data contract. In production, you would store this')
print('alongside the model artifact. Any incoming data that violates it triggers an alert.')

In [ ]:
# Train a baseline model on clean data
# This establishes our "known good" accuracy — our SLO target

X = df[wine.feature_names]
y = df['target']

# Feature engineering: scale features to zero mean, unit variance
# THIS is the preprocessing that must be identical in training and serving
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Train
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
baseline_accuracy = accuracy_score(y_test, model.predict(X_test))
print(f'=== Baseline Model Performance ===')
print(f'Accuracy on clean test data: {baseline_accuracy:.2%}')
print(f'\nThis is our SLO target. If accuracy drops below this in production,')
print(f'something changed in the data — because we did not change the model.')
print(f'\n--- Remember ---')
print(f'Scaler mean (first 3 features): {scaler.mean_[:3].round(3)}')
print(f'Scaler std  (first 3 features): {scaler.scale_[:3].round(3)}')
print(f'These values MUST be used identically in serving. Keep this in mind.')

---
## Part 2 — Writing Data Validation Rules

Now we write validation checks — think of these as your pre-deployment health checks,
but for data instead of services.

In production, these run:
- **Before training:** Is the training data still good?
- **Before inference:** Is the incoming request data valid?
- **On a schedule:** Has the data distribution shifted since last check?

We'll write these as plain pandas checks first (transparent, debuggable),
then note how Great Expectations formalizes the same thing.

In [ ]:
# Data validation functions — your "health check" library for data
# Each function returns (passed: bool, message: str)

def check_no_nulls(dataframe, columns=None):
    """Like a readiness probe: is the data complete?"""
    cols = columns or dataframe.columns
    null_counts = dataframe[cols].isnull().sum()
    failures = null_counts[null_counts > 0]
    if len(failures) == 0:
        return True, 'PASS: No null values found'
    return False, f'FAIL: Nulls found in {dict(failures)}'


def check_value_ranges(dataframe, ranges_dict):
    """Like threshold alerts: are values within expected bounds?"""
    failures = []
    for col, bounds in ranges_dict.items():
        if col not in dataframe.columns:
            failures.append(f'{col}: column missing')
            continue
        below = (dataframe[col] < bounds['min']).sum()
        above = (dataframe[col] > bounds['max']).sum()
        if below > 0 or above > 0:
            failures.append(f'{col}: {below} below min, {above} above max')
    if not failures:
        return True, 'PASS: All values within expected ranges'
    return False, f'FAIL: Out-of-range values in {failures}'


def check_schema(dataframe, expected_columns, expected_dtypes):
    """Like API contract validation: does the shape match the spec?"""
    issues = []
    missing = set(expected_columns) - set(dataframe.columns)
    extra = set(dataframe.columns) - set(expected_columns)
    if missing:
        issues.append(f'Missing columns: {missing}')
    if extra:
        issues.append(f'Unexpected columns: {extra}')
    for col, expected_dtype in expected_dtypes.items():
        if col in dataframe.columns and str(dataframe[col].dtype) != expected_dtype:
            issues.append(f'{col}: expected {expected_dtype}, got {dataframe[col].dtype}')
    if not issues:
        return True, 'PASS: Schema matches contract'
    return False, f'FAIL: Schema violations {issues}'


def check_row_count(dataframe, min_rows):
    """Like a saturation check: do we have enough data?"""
    if len(dataframe) >= min_rows:
        return True, f'PASS: {len(dataframe)} rows (minimum: {min_rows})'
    return False, f'FAIL: Only {len(dataframe)} rows (minimum: {min_rows})'


print('Validation functions defined.')
print('These are your data health checks — same concept as service health checks.')

In [ ]:
# Run all validations on clean data — everything should pass
# This is like running your test suite on a known-good deploy

print('=== Running Data Validation Suite on CLEAN data ===')
print('=' * 55)

checks = [
    ('Null check', check_no_nulls(df, wine.feature_names)),
    ('Value ranges', check_value_ranges(df, EXPECTED_SCHEMA['feature_ranges'])),
    ('Schema match', check_schema(df, EXPECTED_SCHEMA['columns'], EXPECTED_SCHEMA['dtypes'])),
    ('Row count', check_row_count(df, EXPECTED_SCHEMA['row_count_min'])),
]

all_passed = True
for name, (passed, msg) in checks:
    status = '✓' if passed else '✗'
    print(f'  {status} {name}: {msg}')
    if not passed:
        all_passed = False

print('\n' + '=' * 55)
result_msg = 'ALL CHECKS PASSED' if all_passed else 'FAILURES DETECTED'
print(f'Result: {result_msg}')
print('\n--- Operator insight ---')
print('In production, this validation suite runs on every data batch.')
print('Failed checks should: (1) block the pipeline, (2) fire an alert,')
print('(3) log details for investigation. Same as a failed deploy gate.')

---
## Part 3 — Breaking Things: Data Quality Degradation

Now the fun part. We'll introduce realistic data quality issues —
the kind that happen in production when upstream pipelines change —
and observe how they affect model predictions.

Think of this as chaos engineering for data. You break things in staging
to learn what the failure modes look like before they hit production.

In [ ]:
# Scenario 1: Null values introduced (upstream pipeline starts dropping data)
# Real-world cause: A JOIN fails silently, a source system returns empty fields

df_nulls = df.copy()

# Inject 15% nulls into the alcohol column (most important feature)
null_mask = np.random.RandomState(42).random(len(df_nulls)) < 0.15
df_nulls.loc[null_mask, 'alcohol'] = np.nan

# Also inject some nulls into color_intensity
null_mask2 = np.random.RandomState(43).random(len(df_nulls)) < 0.10
df_nulls.loc[null_mask2, 'color_intensity'] = np.nan

print('=== Scenario 1: Null Values Injected ===')
print(f'Nulls in alcohol: {df_nulls["alcohol"].isnull().sum()} '
      f'({df_nulls["alcohol"].isnull().mean():.1%})')
print(f'Nulls in color_intensity: {df_nulls["color_intensity"].isnull().sum()} '
      f'({df_nulls["color_intensity"].isnull().mean():.1%})')

# Run validation
passed, msg = check_no_nulls(df_nulls, wine.feature_names)
print(f'\nValidation result: {msg}')
print('\n--- In production, this alert fires BEFORE the model sees the data ---')
print('Without validation: model receives NaN, might crash or produce garbage.')

In [ ]:
# What happens if we let dirty data through to the model?
# Strategy: Fill nulls with 0 (a common lazy fix in production)

df_filled_zero = df_nulls.copy()
df_filled_zero = df_filled_zero.fillna(0)

# Scale with the SAME scaler (correct approach for serving)
X_dirty = df_filled_zero[wine.feature_names]
X_dirty_scaled = pd.DataFrame(scaler.transform(X_dirty), columns=X_dirty.columns)

# Predict on the full dirty dataset
y_dirty_pred = model.predict(X_dirty_scaled)
dirty_accuracy = accuracy_score(df_filled_zero['target'], y_dirty_pred)

print('=== Impact of Null Values on Model Performance ===')
print(f'Baseline accuracy (clean data):  {baseline_accuracy:.2%}')
print(f'Accuracy with nulls (filled 0):  {dirty_accuracy:.2%}')
print(f'Accuracy DROP:                   {(baseline_accuracy - dirty_accuracy):.2%}')
print()
print('--- Key insight ---')
print('The model did not crash. No exception was thrown. No pod restarted.')
print('But accuracy silently degraded. This is the ML failure mode you must detect.')
print()
print('The validation check we wrote earlier would have CAUGHT this before')
print('the data reached the model. Data validation = your first line of defense.')

In [ ]:
# Scenario 2: Out-of-range values (upstream units change or sensor miscalibration)
# Real-world cause: A field in grams is now in kilograms, or currency conversion applied twice

df_range = df.copy()

# Multiply alcohol values by 10 (simulating a units change)
df_range['alcohol'] = df_range['alcohol'] * 10

# Add extreme outliers to malic_acid (simulating sensor errors)
outlier_idx = np.random.RandomState(44).choice(len(df_range), size=10, replace=False)
df_range.loc[outlier_idx, 'malic_acid'] = 999.0

print('=== Scenario 2: Out-of-Range Values ===')
print(f'Alcohol — Original range: [{df["alcohol"].min():.1f}, {df["alcohol"].max():.1f}]')
print(f'Alcohol — Corrupted range: [{df_range["alcohol"].min():.1f}, {df_range["alcohol"].max():.1f}]')
print(f'Malic acid outliers: {(df_range["malic_acid"] > 100).sum()} values at 999.0')

# Run validation
passed, msg = check_value_ranges(df_range, EXPECTED_SCHEMA['feature_ranges'])
print(f'\nValidation result: {msg}')

# Impact on model
X_range = df_range[wine.feature_names]
X_range_scaled = pd.DataFrame(scaler.transform(X_range), columns=X_range.columns)
y_range_pred = model.predict(X_range_scaled)
range_accuracy = accuracy_score(df_range['target'], y_range_pred)

print(f'\nBaseline accuracy: {baseline_accuracy:.2%}')
print(f'Accuracy with range corruption: {range_accuracy:.2%}')
print(f'Accuracy DROP: {(baseline_accuracy - range_accuracy):.2%}')
print()
print('--- Operator insight ---')
print('A units change is indistinguishable from a bug at the data level.')
print('Range checks catch both. Alerts need BOUNDS, not just null checks.')

In [ ]:
# Scenario 3: Schema change (upstream team renames or drops a column)
# Real-world cause: Database migration, API version change, new data source

df_schema = df.copy()

# Drop a column (team decided 'od280/od315_of_diluted_wines' is deprecated)
df_schema = df_schema.drop(columns=['od280/od315_of_diluted_wines'])

# Rename a column (team standardized naming)
df_schema = df_schema.rename(columns={'color_intensity': 'colour_intensity'})

print('=== Scenario 3: Schema Change ===')
print(f'Original columns: {len(df.columns)}')
print(f'New columns: {len(df_schema.columns)}')
print(f'Missing: od280/od315_of_diluted_wines')
print(f'Renamed: color_intensity -> colour_intensity')

# Run validation
passed, msg = check_schema(df_schema, EXPECTED_SCHEMA['columns'], EXPECTED_SCHEMA['dtypes'])
print(f'\nValidation result: {msg}')

print()
print('--- Operator insight ---')
print('Without schema validation, this causes a KeyError at inference time')
print('(if you are lucky) or silently uses wrong columns (if you are not).')
print('Schema validation catches this IMMEDIATELY at the data boundary.')
print()
print('This is exactly like API contract testing catching breaking changes')
print('before they hit your downstream services.')

---
## Part 4 — Training-Serving Skew (The Silent Killer)

This is the most critical failure mode in production ML.

**Definition:** When the data processing in serving does not exactly match
the data processing used during training.

The model learned patterns from data that was transformed one way.
If you transform serving data differently, the model sees inputs it never
learned from — and gives wrong answers. **No error is thrown.**

This is like config drift between staging and production — everything "works"
but produces wrong results because the environment is subtly different.

In [ ]:
# First, let's see CORRECT serving (no skew)
# The model was trained with StandardScaler. Serving must use the SAME scaler.

# Simulate 20 "production requests" from our test set
sample_raw = df[wine.feature_names].iloc[:20].copy()
sample_labels = df['target'].iloc[:20]

# CORRECT: Use the same scaler that was fitted during training
sample_correct = pd.DataFrame(
    scaler.transform(sample_raw),
    columns=sample_raw.columns
)
predictions_correct = model.predict(sample_correct)
accuracy_correct = accuracy_score(sample_labels, predictions_correct)

print('=== CORRECT Serving (No Skew) ===')
print(f'Using training scaler: mean={scaler.mean_[0]:.3f}, scale={scaler.scale_[0]:.3f} (for alcohol)')
print(f'Accuracy: {accuracy_correct:.2%}')
print(f'Predictions: {list(predictions_correct)}')
print(f'Actual:      {list(sample_labels.values)}')

In [ ]:
# SKEW SCENARIO 1: Different normalization in serving
# Common cause: A serving engineer "simplifies" the preprocessing
# instead of reusing the exact training pipeline.

# WRONG: Fit a NEW scaler on the serving batch (different mean/std)
serving_scaler_wrong = StandardScaler()
sample_skewed_1 = pd.DataFrame(
    serving_scaler_wrong.fit_transform(sample_raw),  # fit on serving data!
    columns=sample_raw.columns
)
predictions_skewed_1 = model.predict(sample_skewed_1)
accuracy_skewed_1 = accuracy_score(sample_labels, predictions_skewed_1)

print('=== SKEW Scenario 1: Re-fitted Scaler ===')
print(f'Training scaler mean (alcohol): {scaler.mean_[0]:.3f}')
print(f'Serving scaler mean (alcohol):  {serving_scaler_wrong.mean_[0]:.3f}  <-- DIFFERENT')
print()
print(f'Correct accuracy:  {accuracy_correct:.2%}')
print(f'Skewed accuracy:   {accuracy_skewed_1:.2%}')
print(f'Accuracy DROP:     {(accuracy_correct - accuracy_skewed_1):.2%}')
print()
print('--- Why this happens ---')
print('The model learned: "when scaled alcohol > 0.5, predict class 0"')
print('But the serving scaler produces different scaled values for the same raw input.')
print('The model now makes decisions based on values it was never trained on.')
print()
print('--- NO ERROR IS THROWN. The pipeline is green. The pod is healthy. ---')

In [ ]:
# SKEW SCENARIO 2: Missing feature transformation in serving
# Common cause: Training notebook applied log transform, serving code forgot it

# Train a model that EXPECTS log-transformed proline
X_train_log = df[wine.feature_names].copy()
X_train_log['proline'] = np.log1p(X_train_log['proline'])  # log transform proline

scaler_with_log = StandardScaler()
X_train_log_scaled = pd.DataFrame(
    scaler_with_log.fit_transform(X_train_log), columns=X_train_log.columns
)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_train_log_scaled, y, test_size=0.2, random_state=42
)
model_log = RandomForestClassifier(n_estimators=100, random_state=42)
model_log.fit(X_tr, y_tr)

# CORRECT serving: apply log transform THEN scale
sample_serve_correct = sample_raw.copy()
sample_serve_correct['proline'] = np.log1p(sample_serve_correct['proline'])
sample_serve_correct_scaled = pd.DataFrame(
    scaler_with_log.transform(sample_serve_correct), columns=sample_raw.columns
)
pred_correct_log = model_log.predict(sample_serve_correct_scaled)

# WRONG serving: forget the log transform, just scale raw values
sample_serve_wrong = sample_raw.copy()  # no log transform!
sample_serve_wrong_scaled = pd.DataFrame(
    scaler_with_log.transform(sample_serve_wrong), columns=sample_raw.columns
)
pred_wrong_log = model_log.predict(sample_serve_wrong_scaled)

acc_correct_log = accuracy_score(sample_labels, pred_correct_log)
acc_wrong_log = accuracy_score(sample_labels, pred_wrong_log)

print('=== SKEW Scenario 2: Forgotten Feature Transform ===')
print(f'Proline value (raw):             {sample_raw["proline"].iloc[0]:.1f}')
print(f'Proline value (log-transformed): {np.log1p(sample_raw["proline"].iloc[0]):.3f}')
print()
print(f'With correct log transform:  accuracy = {acc_correct_log:.2%}')
print(f'Without log transform (skew): accuracy = {acc_wrong_log:.2%}')
print(f'Accuracy DROP: {(acc_correct_log - acc_wrong_log):.2%}')
print()
print('--- This is the most common form of training-serving skew ---')
print('A transformation applied in a training notebook gets forgotten')
print('or reimplemented differently in the serving code.')
print()
print('Prevention: Use the SAME code path for training and serving.')
print('Detection: Compare feature distributions between training data')
print('and live serving data. If they diverge, you have skew.')

In [ ]:
# Visualize the impact — this is what your monitoring dashboard would show

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Accuracy comparison across all scenarios
scenarios = ['Clean\nbaseline', 'Nulls\n(filled 0)', 'Range\ncorruption',
             'Skew:\nre-fit', 'Skew:\nno log']
accuracies = [baseline_accuracy, dirty_accuracy, range_accuracy,
              accuracy_skewed_1, acc_wrong_log]
colors = ['green' if a > 0.9 else 'orange' if a > 0.7 else 'red' for a in accuracies]

axes[0].bar(scenarios, accuracies, color=colors, edgecolor='black', alpha=0.7)
axes[0].axhline(y=baseline_accuracy, color='green', linestyle='--', alpha=0.5,
                label='SLO target')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy by Data Issue')
axes[0].set_ylim(0, 1.05)
axes[0].legend()

# Plot 2: Feature distribution — correct vs skewed
axes[1].hist(sample_correct['alcohol'], bins=10, alpha=0.6,
             label='Correct scaling', color='green')
axes[1].hist(sample_skewed_1['alcohol'], bins=10, alpha=0.6,
             label='Wrong scaling', color='red')
axes[1].set_xlabel('Scaled alcohol value')
axes[1].set_title('Feature Distribution:\nCorrect vs Skewed')
axes[1].legend()

# Plot 3: Prediction confidence — model can be confident AND wrong
conf_correct = model.predict_proba(sample_correct).max(axis=1)
conf_skewed = model.predict_proba(sample_skewed_1).max(axis=1)
axes[2].hist(conf_correct, bins=10, alpha=0.6, label='Correct', color='green')
axes[2].hist(conf_skewed, bins=10, alpha=0.6, label='Skewed', color='red')
axes[2].set_xlabel('Prediction confidence')
axes[2].set_title('Confidence: Model can be\nconfident AND wrong')
axes[2].legend()

plt.tight_layout()
plt.savefig('data_quality_impact.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n--- What this chart tells you ---')
print('1. Data quality issues cause accuracy drops WITHOUT throwing errors')
print('2. The model can be confident AND wrong (right chart)')
print('3. You need data validation + distribution monitoring to cover full risk')

---
## Part 5 — Feature Engineering in Practice

Let's see a quick example of feature engineering to make the concept concrete.
We'll transform raw data into features, and see how it affects model performance.

In [ ]:
# Feature engineering example: creating new features from existing ones
# This is what data scientists do — and what you need to replicate in serving

df_features = df.copy()

# Create derived features (common in production)
# Ratio features — these capture relationships between raw columns
df_features['alcohol_to_malic'] = df_features['alcohol'] / (df_features['malic_acid'] + 0.01)
df_features['color_x_proline'] = df_features['color_intensity'] * df_features['proline']
df_features['flavanoids_ratio'] = df_features['flavanoids'] / (df_features['nonflavanoid_phenols'] + 0.01)

# Log transforms for skewed features
df_features['log_proline'] = np.log1p(df_features['proline'])
df_features['log_magnesium'] = np.log1p(df_features['magnesium'])

new_features = ['alcohol_to_malic', 'color_x_proline', 'flavanoids_ratio',
                'log_proline', 'log_magnesium']

print('=== Feature Engineering Example ===')
print(f'Original features: {len(wine.feature_names)}')
print(f'New engineered features: {len(new_features)}')
print(f'Total features: {len(wine.feature_names) + len(new_features)}')
print()
print('Engineered features:')
for feat in new_features:
    print(f'  {feat}: range [{df_features[feat].min():.2f}, {df_features[feat].max():.2f}]')

# Compare model with and without engineered features
all_features = list(wine.feature_names) + new_features
X_eng = df_features[all_features]
scaler_eng = StandardScaler()
X_eng_scaled = pd.DataFrame(scaler_eng.fit_transform(X_eng), columns=all_features)
X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(
    X_eng_scaled, y, test_size=0.2, random_state=42
)
model_eng = RandomForestClassifier(n_estimators=100, random_state=42)
model_eng.fit(X_tr_e, y_tr_e)
acc_eng = accuracy_score(y_te_e, model_eng.predict(X_te_e))

print(f'\nAccuracy without feature engineering: {baseline_accuracy:.2%}')
print(f'Accuracy with feature engineering:    {acc_eng:.2%}')
print()
print('--- Operator insight ---')
print('Every one of these transforms becomes serving code that MUST run')
print('on every incoming request. If any transform is wrong or missing,')
print('you have training-serving skew. This is why feature stores exist —')
print('they ensure the same transformation code runs in both training and serving.')

---
## Part 6 — Mapping Validations to Production Alerts

Everything we've done maps directly to production monitoring.
Here's the translation table from "lab exercise" to "production alert you'd configure."

In [ ]:
# Summary: Data validation → Production alert mapping

alert_mapping = pd.DataFrame({
    'Validation Check': [
        'Null value check',
        'Value range check',
        'Schema validation',
        'Row count check',
        'Distribution shift (skew detection)',
        'Feature transform verification',
    ],
    'Production Alert': [
        'null_rate > 1% on any feature',
        'feature_value outside [min, max] bounds',
        'column_count != expected OR dtype mismatch',
        'batch_size < minimum_threshold',
        'KL-divergence(serving, training) > threshold',
        'feature_stats(serving) != feature_stats(training)',
    ],
    'Severity': [
        'P2 - Block pipeline',
        'P2 - Block pipeline',
        'P1 - Immediate page',
        'P3 - Warning, investigate',
        'P2 - Investigate within 4h',
        'P1 - Immediate page (skew)',
    ],
    'Response Action': [
        'Halt inference, check upstream data source',
        'Halt inference, check for units/schema change',
        'Halt pipeline, roll back to last known schema',
        'Check data source availability',
        'Compare distributions, check for skew',
        'Verify preprocessing code matches training',
    ],
    'SLO Example': [
        'null_rate < 0.1% per feature per hour',
        '99.9% of values within trained bounds',
        '100% schema match (zero tolerance)',
        'batch_size >= 80% of expected',
        'KL-div < 0.1 on rolling 1h window',
        'feature mean within 2 std of training mean',
    ],
})

print('=== Data Validation → Production Alert Mapping ===')
print()
print(alert_mapping.to_string(index=False))
print()
print('--- How to use this table ---')
print('1. Each row becomes a Prometheus/CloudWatch metric + alert rule')
print('2. Severity maps to your existing incident priority framework')
print('3. Response actions go in your ML runbook (Week 10)')
print('4. SLOs get tracked on your dashboard (Week 9)')
print()
print('This is YOUR territory — alerting, SLOs, runbooks. Now you know')
print('what the ML-specific signals are that feed into them.')

---
## Bonus: Great Expectations (Production-Grade Validation)

The manual pandas checks above are great for learning and lightweight scripts.
In production, teams use **Great Expectations** (GE) to:

- Define expectations as code (version-controlled, reviewable)
- Generate data documentation automatically
- Run validation suites in pipelines with pass/fail gates
- Track validation results over time (data quality SLO dashboards)

Think of GE as the "pytest for data" — same concept as your test suites,
but assertions are about data properties instead of code behavior.

In [ ]:
# Great Expectations — the production-grade version of our manual checks
# This shows the API style. Same concepts, more infrastructure.

try:
    import great_expectations as gx
    from great_expectations.dataset import PandasDataset

    # Wrap our DataFrame in a GE dataset
    ge_df = PandasDataset(df)

    # Define expectations (same checks as before, but in GE's declarative style)
    print('=== Great Expectations Validation ===')
    print()

    # Null checks
    result = ge_df.expect_column_values_to_not_be_null('alcohol')
    print(f'alcohol not null: {"PASS" if result.success else "FAIL"}')

    # Range checks
    result = ge_df.expect_column_values_to_be_between('alcohol', min_value=10, max_value=15)
    print(f'alcohol in [10, 15]: {"PASS" if result.success else "FAIL"}')

    # Type checks
    result = ge_df.expect_column_values_to_be_of_type('alcohol', 'float64')
    print(f'alcohol is float64: {"PASS" if result.success else "FAIL"}')

    # Column presence
    for col in wine.feature_names[:3]:
        result = ge_df.expect_column_to_exist(col)
        print(f'{col} exists: {"PASS" if result.success else "FAIL"}')

    print()
    print('--- In production, GE runs as a pipeline step ---')
    print('If any expectation fails, the pipeline halts and alerts fire.')
    print('Results are stored for audit trail (important in banking).')

except ImportError:
    print('Great Expectations not installed. Run: pip install great-expectations')
    print('The manual pandas checks above cover the same concepts.')
    print('GE adds: declarative syntax, data docs, pipeline integration, audit logs.')

---
## Summary — What You Just Learned

| What we did | ML term | Ops analogy | Production implementation |
|-------------|---------|-------------|---------------------------|
| Recorded healthy data stats | Schema/contract definition | OpenAPI spec | Store with model artifact |
| Wrote null checks | Data completeness validation | Readiness probe | Pre-inference gate |
| Wrote range checks | Data range validation | Threshold alerting | Prometheus metric + alert |
| Wrote schema checks | Schema validation | API contract test | Pipeline gate (block on fail) |
| Broke data with nulls | Data quality degradation | Upstream dependency failure | Alert: null_rate SLO breach |
| Broke data with range issues | Feature distribution shift | Config drift | Alert: value_range SLO breach |
| Demonstrated skew (re-fit) | Training-serving skew | Env config mismatch | Compare train vs serve stats |
| Demonstrated skew (missing transform) | Training-serving skew | Missing deployment step | Feature transform verification |
| Mapped checks to alerts | ML observability design | Alert framework design | Dashboard + runbook |

## Key Takeaways for an Operator

1. **Data bugs cause model bugs.** 80% of ML production issues trace to data, not models.
2. **Schema validation is your first defense.** Catches breaking changes at the boundary.
3. **Training-serving skew is the #1 silent failure mode.** Same code must run in both paths.
4. **Every validation maps to an alert.** You already know how to build alerting — now you know what to alert on.
5. **The model never throws an error when data is wrong.** It just gives wrong answers confidently.

## What's Next

- **Week 3:** We'll detect drift — when data changes slowly over time (not a sudden break,
  but a gradual shift). This is the subtle version of what we broke manually today.
- **Week 9:** We'll turn these validations into real SLOs with Prometheus alerts.